# **Estrategias de Inversión**
---

Existen 2 tipos de estrategias de inversión:

- **Pasiva**: Es una estrategia a largo plazo, busca replicar el desempeño de un <u>índice o sector de referencia</u>.  
    - Menor monitoreo 
    - Menor costo
    - Menor riesgo en el largo plazo
    - Menor carga fiscal
    - Buena duversificación por sí sola
- **Activa**: Cualquier tipo de estrategia <u>donde tú buscas seleccionar acciones</u> (stock picking), busca superar el rendimiento de un índice de referencia o del mercado en general.
    - Constante análisis y ajustes en un portafolio con objetivo de generar retornos superiores a los de tu *benchmark*
    - Nivel más alto de involucramiento y toma de decisiones por parte del gestor
    - Comisiones más altas para el inversionista
    - Debe ser evaluada contra un *benchmark*


Un benchmark es un índice de referencia que se usa para medir y comparar el desempeño de una estrategia activa. Representa el comportamiento promedio de un grupo de activos o un mercado específico. Ayuda a evaluar la eficacia y justificar la estrategia seguida por un gestor de portafolios, ayudando a determinar al inversionista si está generando valor por encima del mercado. Su importancia radica en:

- Medición de desempeño
- Análisis de alpha
- Gestión y análisis del riesgo
        
## Comparando el desempeño

Consideremos el $\text{Radio de Sharpe}$, que es el rendimiento que el portafolio obtuvo por encima de la tasa libre de riesgo $r_f$ ajustado por el riesgo asumido, su fórmula es la siguiente:

$$
RS = \frac{R_p - r_f}{\sigma_p}
$$

Mientras **mayor sea el Ratio de Sharpe, mejor es la relación rendimiento-riesgo**.

Como referencia general:

| Ratio de Sharpe | Interpretación |
|---|---|
| $< 0$ | Malo: el portafolio rindió menos que el activo libre de riesgo |
| $0 - 1$ | Relación rendimiento-riesgo relativamente baja |
| $1 - 2$ | Buena |
| $2 - 3$ | Muy buena |
| $> 3$ | Excelente, aunque poco común de forma sostenida |

Lo importante es que **no debe analizarse solamente el rendimiento**. Por ejemplo, si tienes:

- **Portafolio A:** rendimiento = $15\%$, volatilidad = $20\%$, Sharpe = $0.50$
- **Portafolio B:** rendimiento = $12\%$, volatilidad = $8\%$, Sharpe = $0.88$

Aunque **A gana más, B administra mejor el riesgo**, porque obtiene más rendimiento excedente por cada unidad de volatilidad asumida.

Y la $\text{Aversión al Riesgo}$, que es una medida relativa que refleja la preferencia de un inversionista por estrategias con menor riesgo, priorizando la estabilidad sobre la posibilidad de obtener mayores rendimientos.

Hay tres perfiles de riesgo:

- **Amante al Riesgo:** Prefiere mayor riesgo y mayor rendimiento.
- **Neutral al Riesgo:** Le da igual el riesgo siempre y cuando la relación entre riesgo y rendimiento sea buena.
- **Averso al Riesgo:** Prefiere el menor riesgo posible.

Se calcula con la siguiente fórmula:
$$
AR = \frac{R_p - r_f}{\frac{1}{2}\sigma_p^2}
$$

Pero de momento, solo nos quedaremos con la fórmula del $\text{Radio de Sharpe}$.

-----

Ahora, mediante esta métrica ($RS$), evaluaremos dos portafolios, el primero con una estrategia pasiva y el segundo con una activa.

Importamos las librerías

In [1]:
import yfinance as yf
import numpy as np
import pandas as pd

Definimos los tickers con los que vamos a trabajar (separados entre estrategias).

In [2]:
active_tickers = ['COHR', 'KO', 'SNOW', 'MRK']
passive_tickers = ['^GSPC']  # Índice S&P 500

tickers = active_tickers + passive_tickers

Descargamos los precios de cierre.

In [3]:
prices = yf.download(tickers, start='2010-01-01', end='2026-09-01')['Close']
prices

[*********************100%***********************]  5 of 5 completed


Ticker,COHR,KO,MRK,SNOW,^GSPC
Date,,,,,
2010-01-04,16.080000,17.205252,20.314669,NaN,1132.989990
2010-01-05,16.285000,16.997126,20.397007,NaN,1136.520020
2010-01-06,16.385000,16.991091,20.671459,NaN,1137.140015
2010-01-07,15.740000,16.948856,20.704393,NaN,1141.689941
2010-01-08,15.675000,16.635160,20.693417,NaN,1144.979980
...,...,...,...,...,...
2026-08-25,288.140015,91.639999,156.449997,317.019989,7677.279785
2026-08-26,294.369995,90.080002,153.100006,315.369995,7675.700195
2026-08-27,295.390015,89.059998,149.539993,329.109985,7730.990234


Calculamos los rendimientos diarios.

In [4]:
rets = prices.pct_change().dropna()
rets

Ticker,COHR,KO,MRK,SNOW,^GSPC
Date,,,,,
2020-09-17,-0.018394,-0.004725,0.000818,-0.103926,-0.008412
2020-09-18,0.008709,-0.001978,0.001985,0.054760,-0.011183
2020-09-21,0.007849,-0.026958,-0.031232,-0.046458,-0.011571
2020-09-22,0.057892,0.011612,-0.002285,0.027573,0.010518
2020-09-23,-0.052025,-0.029198,-0.003738,-0.075566,-0.023721
...,...,...,...,...,...
2026-08-25,0.045918,-0.003805,0.038431,-0.017845,0.003191
2026-08-26,0.021621,-0.017023,-0.021413,-0.005205,-0.000206
2026-08-27,0.003465,-0.011323,-0.023253,0.043568,0.007203


Vamos a definir desde este momento, la tasa libre de riesgo $r_f$, para ello, como los activos son estadounidenses, usaremos la tasa de los bonos del tesoro que, a 2026-09-02, está alrededor del 4%.

In [5]:
rf = 0.04 # Bonos de EUA

### Estrategia Pasiva

Para esta estrategia usaremos el índice del `S&P 500`.

In [6]:
rets_passive = rets[passive_tickers]
rets_passive

Ticker,^GSPC
Date,
2020-09-17,-0.008412
2020-09-18,-0.011183
2020-09-21,-0.011571
2020-09-22,0.010518
2020-09-23,-0.023721
...,...
2026-08-25,0.003191
2026-08-26,-0.000206
2026-08-27,0.007203


Calculamos el Rendimiento Esperado (anualizado).

In [7]:
E_R_passive = rets_passive.mean() * 252
E_R_passive

Ticker
^GSPC    0.152006
dtype: float64

Ahora, su volatilidad (también anualizada).

In [8]:
S_D_passive = rets_passive.std() * np.sqrt(252)
S_D_passive

Ticker
^GSPC    0.166016
dtype: float64

Calculamos Radio de Sharpe.

In [9]:
Sharpe_Ratio_passive = (E_R_passive - rf) / S_D_passive  # Benchmark
Sharpe_Ratio_passive 

Ticker
^GSPC    0.67467
dtype: float64

Este sería nuestro $Benchmark$, y se interpreta de la siguiente manera:

Por cada unidad de riesgo asumida, obtengo 0.67 unidades de rendimiento, en exceso a $r_f$.

¿Es bueno o malo?

Bueno, necesitamos compararlo con el siguiente.

### Estrategia Activa

Para esta estrategia, escogeremos los siguientes 4 activos: `GOHR`, `KO`, `SNOW` y `MRK`.

Además, definimos los pesos de cada activo, en este caso será un portafolio equiponderado.

In [10]:
weights = np.ones(4) / 4  # Pesos iguales para las 4 acciones

In [11]:
rets_active = rets[active_tickers]
rets_active

Ticker,COHR,KO,SNOW,MRK
Date,,,,
2020-09-17,-0.018394,-0.004725,-0.103926,0.000818
2020-09-18,0.008709,-0.001978,0.054760,0.001985
2020-09-21,0.007849,-0.026958,-0.046458,-0.031232
2020-09-22,0.057892,0.011612,0.027573,-0.002285
2020-09-23,-0.052025,-0.029198,-0.075566,-0.003738
...,...,...,...,...
2026-08-25,0.045918,-0.003805,-0.017845,0.038431
2026-08-26,0.021621,-0.017023,-0.005205,-0.021413
2026-08-27,0.003465,-0.011323,0.043568,-0.023253


Calculamos el rendimiento esperado (anualizado).

In [12]:
means_active = rets_active.mean() * 252
E_R_active = means_active@weights
E_R_active

np.float64(0.26454335414888175)

Ahora de nuevo, la volatilidad (anualizada).

In [13]:
cov_matrix_active = rets_active.cov()
cov_matrix_active

Ticker,COHR,KO,SNOW,MRK
Ticker,,,,
COHR,0.001551,-0.000021,0.000403,0.000007
KO,-0.000021,0.000112,-0.000003,0.000047
SNOW,0.000403,-0.000003,0.001525,-0.000015
MRK,0.000007,0.000047,-0.000015,0.000226


In [14]:
vol_R_active = np.sqrt(weights.T @ cov_matrix_active @ weights) * np.sqrt(252)
vol_R_active

np.float64(0.2587503844756821)

Por último, el Radio de Sharpe.

In [15]:
Sharpe_Ratio_active = (E_R_active - rf) / vol_R_active
Sharpe_Ratio_active

np.float64(0.8677991130482158)

Por lo que vemos, en este caso el Radio de Sharpe de la estrategia activa fue mayor, por lo que podemos concluir que para este caso, nos conviene una estrategia de inversión activa.